In [1]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel

torch.cuda.empty_cache()

# model_name = "meta-llama/Llama-3.2-1B"
model_name = "Qwen/Qwen3-0.6B"
model_save_dir = "model/downloaded"
adapter_save_dir = "model/downloaded/qwen3_0.6b_med_3eps_test"
os.makedirs(model_save_dir, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="cuda",
    cache_dir=model_save_dir
)
peft_model = PeftModel.from_pretrained(
    model,
    adapter_save_dir)

In [ ]:
from pruning.prune import foresight_prune, prune_wanda
from data.datasets import get_loaders

dataloader, _ = get_loaders(
    name="c4",
    tokenizer=tokenizer,
    seqlen=2048,
    nsamples=128)

foresight_prune(model=peft_model,
                dataloader=dataloader,
                prune_ratio=0.5,
                mask_lr=0.5,
                nsamples=128)

# model = peft_model.merge_and_unload()
# prune_wanda(sparsity_ratio=0.5,
#             nsamples=128,
#             seed=42,
#             seqlen=2048,
#             model=model,
#             tokenizer=tokenizer)
            

Token indices sequence length is longer than the specified maximum sequence length for this model (528225 > 131072). Running this sequence through the model will result in indexing errors


preparing calibration input


In [ ]:
from pruning.utils import check_sparsity
model = peft_model.merge_and_unload()
sparsity = check_sparsity(model)
print(f"Sparsity after pruning: {sparsity:.2%}")

layer 0 sparsity 0.399544
layer 1 sparsity 0.399544
layer 2 sparsity 0.399545
layer 3 sparsity 0.399544
layer 4 sparsity 0.399544
layer 5 sparsity 0.399544
layer 6 sparsity 0.399544
layer 7 sparsity 0.399544
layer 8 sparsity 0.399544
layer 9 sparsity 0.399544
layer 10 sparsity 0.399544
layer 11 sparsity 0.399544
layer 12 sparsity 0.399544
layer 13 sparsity 0.399544
layer 14 sparsity 0.399544
layer 15 sparsity 0.399544
layer 16 sparsity 0.399544
layer 17 sparsity 0.399544
layer 18 sparsity 0.399544
layer 19 sparsity 0.399544
layer 20 sparsity 0.399544
layer 21 sparsity 0.399544
layer 22 sparsity 0.399544
layer 23 sparsity 0.399544
layer 24 sparsity 0.399544
layer 25 sparsity 0.399544
layer 26 sparsity 0.399544
layer 27 sparsity 0.399544
Sparsity after pruning: 39.95%


In [ ]:
from evaluation.perplexity import eval_ppl

ppl = eval_ppl(model = model, tokenizer=tokenizer, dataset="harrison", seqlen=2048)
print (f"Perplexity after pruning: {ppl}")

evaluating on harrison


block 0/300
block 50/300
block 100/300
block 150/300
block 200/300
block 250/300
Perplexity after pruning: 22.601875592776874


In [ ]:
from evaluation.domain_zero_shot import evaluate_pubmedqa, evaluate_mednli, evaluate_hqs

post = {}
acc, maf, cm, predictions = evaluate_pubmedqa(model, tokenizer)
print(f"PubMedQA Accuracy after fine-tuning: {acc:.4f}, Macro F1: {maf:.4f}")
post["PubMedQA"] = {"accuracy": float(acc), "macro_f1": float(maf)}
acc, maf, cm, predictions = evaluate_mednli(model, tokenizer)
print(f"MedNLI Accuracy after fine-tuning: {acc:.4f}, Macro F1: {maf:.4f}")
post["MedNLI"] = {"accuracy": float(acc), "macro_f1": float(maf)}
rouge, prediction = evaluate_hqs(model, tokenizer)
print("HQS ROUGE after fine-tuning:", rouge)
post["HQS"] = {"rouge": rouge}

Predicting final decisions: 100%|██████████| 500/500 [01:23<00:00,  5.99it/s]


=== PubMedQA Results ===
Saved detailed log to: /home/918839576/Trepo/Efficient_Domain_Adaptation/assets/pubmedqa_predictions.txt
Accuracy = 0.6660
Macro-F1 = 0.4608
Macro-F1 95% bootstrap CI (n_boot=1000): [0.4311, 0.4897]
PubMedQA Accuracy after fine-tuning: 0.6660, Macro F1: 0.4608


Evaluating MedNLI: 100%|██████████| 1422/1422 [03:58<00:00,  5.96it/s]


=== MedNLI Results ===
Saved detailed log to: /home/918839576/Trepo/Efficient_Domain_Adaptation/assets/mednli_predictions.txt
Accuracy = 0.6371
Macro-F1 = 0.6396
Macro-F1 95% bootstrap CI (n_boot=1000): [0.6149, 0.6642]
MedNLI Accuracy after fine-tuning: 0.6371, Macro F1: 0.6396


Generating Summaries: 100%|██████████| 100/100 [01:22<00:00,  1.21it/s]


=== Final ROUGE SCORES ===
ROUGE-1: 0.3125
ROUGE-2: 0.1123
ROUGE-L: 0.2867

Total generation and evaluation time: 82.82 seconds
HQS ROUGE after fine-tuning: {'rouge1': np.float64(0.31245091758407073), 'rouge2': np.float64(0.1123008409930543), 'rougeL': np.float64(0.28672017245729153), 'rougeLsum': np.float64(0.28616141823227437)}
